# 03. 심화: 불확실성·OOD·인과 A/B 평가

## 학습 목표

- 부트스트랩 앙상블의 예측 분산과 입력 거리로 불확실성/OOD(out-of-distribution)를 탐지한다.
- 고위험 예측을 보류(abstention)할 때 정확도뿐 아니라 coverage도 함께 보고한다.
- 무작위 A/B 실험에서 전환율 차이와 부트스트랩 신뢰구간을 계산한다.
- 표본 크기에 따른 검정력을 모의실험하고, 관찰 로그의 교란 편향을 확인한다.
- 개인정보·공정성·운영 실패를 포함한 배포 체크리스트를 작성한다.

> **Toy reproduction 주의:** 모든 자료와 결과는 교육용 합성 실험이다. 논문 및 후속 공개 모델·데이터 아티팩트를 사용하지 않으며, 논문이 보고한 96.7% 정확도·43.2% 전환 향상 또는 실제 사업 효과를 재현하지 않는다.

In [ ]:
import numpy as np

SEED = 250323303
rng = np.random.default_rng(SEED)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))

def log_loss(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def brier(y, p):
    return np.mean((np.asarray(y) - np.asarray(p)) ** 2)

def accuracy(y, p):
    return np.mean((np.asarray(p) >= 0.5) == np.asarray(y))

def fit_logistic(X, y, steps=650, lr=0.08, l2=2e-3):
    w = np.zeros(X.shape[1])
    penalty = np.r_[0.0, np.ones(X.shape[1] - 1)]
    for _ in range(steps):
        p = sigmoid(X @ w)
        gradient = X.T @ (p - y) / len(y) + l2 * penalty * w
        w -= lr * gradient
    return w

print("NumPy", np.__version__, "| seed", SEED)

## 1. 앙상블 불확실성과 OOD 보류

학습 도메인은 일반적인 관여도·이의제기 분포라고 가정한다. 평가 자료 일부에는 입력 분포 이동과 함께 관계 자체가 달라지는 개념 이동을 의도적으로 넣는다. 부트스트랩 모델 간 표준편차는 매개변수 불확실성의 거친 신호이고, Mahalanobis 거리는 학습 입력에서 얼마나 멀리 벗어났는지를 나타낸다. 둘 다 안전을 보장하는 완전한 방법은 아니다.

In [ ]:
def make_domain(n, rng, ood=False):
    engagement = rng.normal(loc=2.2 if ood else 0.0, scale=1.0, size=n)
    objections = rng.normal(loc=-1.4 if ood else 0.0, scale=1.0, size=n)
    group = rng.binomial(1, 0.45, size=n).astype(float)
    X = np.column_stack([engagement, objections, group])
    if ood:
        # 교육용 concept shift: 학습 도메인의 관계를 그대로 외삽하면 실패하도록 만든다.
        logit = -0.2 - 0.75 * engagement - 0.55 * objections - 0.35 * group
    else:
        logit = -0.8 + 0.95 * engagement - 0.70 * objections - 0.35 * group
    y = rng.binomial(1, sigmoid(logit)).astype(float)
    return X, y

X_train_raw, y_train = make_domain(1800, rng, ood=False)
X_id_raw, y_id = make_domain(900, rng, ood=False)
X_ood_raw, y_ood = make_domain(450, rng, ood=True)
X_eval_raw = np.vstack([X_id_raw, X_ood_raw])
y_eval = np.r_[y_id, y_ood]
is_ood = np.r_[np.zeros(len(y_id), dtype=bool), np.ones(len(y_ood), dtype=bool)]

mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0) + 1e-8
X_train = np.column_stack([np.ones(len(X_train_raw)), (X_train_raw - mean) / std])
X_eval = np.column_stack([np.ones(len(X_eval_raw)), (X_eval_raw - mean) / std])

ensemble = []
for _ in range(12):
    index = rng.integers(0, len(y_train), size=len(y_train))
    ensemble.append(fit_logistic(X_train[index], y_train[index]))
predictions = np.column_stack([sigmoid(X_eval @ w) for w in ensemble])
mean_probability = predictions.mean(axis=1)
ensemble_std = predictions.std(axis=1)

# 표준화 공간의 공분산으로 입력 거리를 계산한다. pseudo-inverse는 수치적 안전장치다.
train_features = X_train[:, 1:]
inverse_covariance = np.linalg.pinv(np.cov(train_features, rowvar=False))
eval_features = X_eval[:, 1:]
mahalanobis = np.sqrt(np.einsum("ij,jk,ik->i", eval_features, inverse_covariance, eval_features))
combined_uncertainty = ensemble_std / (np.median(ensemble_std) + 1e-8) + mahalanobis
cutoff = np.quantile(combined_uncertainty, 0.70)
accepted = combined_uncertainty <= cutoff

def report(name, mask):
    print(
        f"{name:18s} n={mask.sum():4d}, accuracy={accuracy(y_eval[mask], mean_probability[mask]):.3f}, "
        f"log-loss={log_loss(y_eval[mask], mean_probability[mask]):.3f}, "
        f"Brier={brier(y_eval[mask], mean_probability[mask]):.3f}"
    )

report("전체 평가", np.ones(len(y_eval), dtype=bool))
report("ID만", ~is_ood)
report("OOD만", is_ood)
report("불확실성 보류 후", accepted)
print(f"coverage={accepted.mean():.3f}, OOD 보류율={(~accepted & is_ood).sum() / is_ood.sum():.3f}")

assert np.all((mean_probability >= 0.0) & (mean_probability <= 1.0))
assert 0.68 <= accepted.mean() <= 0.72
assert np.mean(mahalanobis[is_ood]) > np.mean(mahalanobis[~is_ood])
assert log_loss(y_eval[accepted], mean_probability[accepted]) < log_loss(y_eval, mean_probability)
print("검증 통과: OOD 거리가 증가했고, 보류 후 coverage와 품질을 함께 보고했다.")

### 결과 해석

보류 후 지표가 좋아져도 이는 선택된 쉬운 사례에서의 성능이다. 따라서 반드시 `coverage`, 보류된 고객의 구성, 그룹별 보류율과 사람 검토 결과를 함께 공개해야 한다. 앙상블 합의가 높더라도 모든 모델이 같은 잘못된 외삽을 할 수 있으므로 OOD 탐지와 사후 모니터링이 별도로 필요하다.

## 2. 무작위 A/B 실험과 불확실성

전환율 향상은 예측 정확도와 다른 인과 질문이다. 아래에서는 고객을 처리군/대조군에 무작위 배정하고, 두 전환율의 차이를 부트스트랩한다. 신뢰구간은 이번 표본의 불확실성을 나타내며 실제 운영의 비순응, 간섭, 중도탈락을 자동으로 해결하지는 않는다.

In [ ]:
experiment_rng = np.random.default_rng(SEED + 10)
n = 6000
customer_score = experiment_rng.normal(size=n)
treatment = experiment_rng.binomial(1, 0.5, size=n)
baseline_logit = -1.25 + 0.65 * customer_score
# 확률 척도의 효과는 고객 상태에 따라 달라진다.
outcome_probability = sigmoid(baseline_logit + treatment * (0.38 + 0.12 * customer_score))
outcome = experiment_rng.binomial(1, outcome_probability)

control_rate = outcome[treatment == 0].mean()
treatment_rate = outcome[treatment == 1].mean()
uplift = treatment_rate - control_rate

bootstrap_rng = np.random.default_rng(SEED + 11)
bootstrap_uplift = np.empty(1500)
control_outcomes = outcome[treatment == 0]
treatment_outcomes = outcome[treatment == 1]
for b in range(len(bootstrap_uplift)):
    control_sample = bootstrap_rng.choice(control_outcomes, size=len(control_outcomes), replace=True)
    treatment_sample = bootstrap_rng.choice(treatment_outcomes, size=len(treatment_outcomes), replace=True)
    bootstrap_uplift[b] = treatment_sample.mean() - control_sample.mean()
ci_low, ci_high = np.quantile(bootstrap_uplift, [0.025, 0.975])
print(f"대조군 전환율={control_rate:.3f}, 처리군 전환율={treatment_rate:.3f}")
print(f"절대 uplift={uplift:.3f}, bootstrap 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")

assert abs(treatment.mean() - 0.5) < 0.03
assert ci_low < uplift < ci_high
assert np.all(np.isfinite(bootstrap_uplift))

## 3. 표본 크기와 관찰 로그의 교란

같은 효과라도 표본이 작으면 신뢰구간이 넓고 검정력이 낮다. 아래 power는 특정 합성 가정에서만 유효한 Monte Carlo 감도 분석이다. 이어지는 관찰 연구 예시는 영업 담당자가 고위험 고객에게 처리를 더 자주 주면 단순 처리군 비교가 왜 편향되는지를 보여준다.

In [ ]:
def approximate_power(total_n, simulations=500, p_control=0.22, p_treatment=0.28, seed=0):
    local_rng = np.random.default_rng(seed)
    significant = 0
    half = total_n // 2
    for _ in range(simulations):
        c = local_rng.binomial(1, p_control, size=half)
        t = local_rng.binomial(1, p_treatment, size=total_n - half)
        difference = t.mean() - c.mean()
        standard_error = np.sqrt(c.mean() * (1 - c.mean()) / len(c) + t.mean() * (1 - t.mean()) / len(t))
        significant += difference > 1.96 * standard_error
    return significant / simulations

powers = {}
for total_n in [400, 1000, 3000]:
    powers[total_n] = approximate_power(total_n, seed=SEED + total_n)
    print(f"총 표본 {total_n:4d}: 양측 5% 기준 근사 검정력={powers[total_n]:.3f}")

obs_rng = np.random.default_rng(SEED + 20)
risk = obs_rng.normal(size=10000)
# 위험도가 높은 고객일수록 개입을 더 자주 받으며, 위험도 자체도 전환에 영향을 준다.
observed_treatment = obs_rng.binomial(1, sigmoid(1.25 * risk))
observed_outcome = obs_rng.binomial(1, sigmoid(-1.1 + 0.85 * risk + 0.35 * observed_treatment))
naive_difference = observed_outcome[observed_treatment == 1].mean() - observed_outcome[observed_treatment == 0].mean()
print(f"교란된 관찰 로그의 단순 전환율 차이={naive_difference:.3f}")
print("이 차이는 처리 효과와 고객 위험도 차이가 섞여 있어 인과 효과가 아니다.")

assert powers[3000] > powers[400]
assert np.isfinite(naive_difference)
print("검증 통과: 더 큰 표본에서 검정력이 높아졌고 교란 예시를 계산했다.")

## 배포 전 개인정보·공정성·운영 체크리스트

### 개인정보와 보안

- [ ] 녹취·채팅 수집에 적법한 근거와 고지/동의가 있는가? 보존 기간과 삭제 절차가 정해졌는가?
- [ ] 이름, 연락처, 결제·건강 정보 등 민감 데이터를 임베딩/API로 보내기 전에 최소화·마스킹하는가?
- [ ] 원문, 임베딩, 예측, 상담원 행동 로그의 접근 제어·암호화·감사를 분리했는가?
- [ ] 외부 모델 공급자의 학습 사용 여부, 지역 저장, 침해 대응과 데이터 처리 계약을 확인했는가?

### 공정성과 고객 피해

- [ ] 보호집단과 교차집단별 accuracy, calibration, 오탐/미탐, 보류율과 실제 혜택을 측정하는가?
- [ ] 할인·압박 전략이 취약 고객에게 집중되거나 대리변수로 차별을 만들지 않는가?
- [ ] 상담원과 고객이 자동화 개입을 알 수 있고, 이의 제기·사람 검토·opt-out 경로가 있는가?

### 평가와 운영

- [ ] 대화/고객/시간 단위 누수를 막은 외부 검증과 확률 보정을 수행했는가?
- [ ] 전환 외에 환불, 불만, 장기 유지, 마진과 고객 만족을 guardrail로 두었는가?
- [ ] A/B 배정, 사전 정의한 지표, 표본 크기, 중도탈락과 다중 비교를 기록했는가?
- [ ] OOD·drift·latency·결측·API 장애 시 보류 또는 안전한 기본 정책으로 전환하는가?
- [ ] 모델/프롬프트/임베딩 버전, 입력 스키마와 의사결정 경로를 재현 가능하게 남기는가?

핵심 원칙은 ‘높은 예측 점수’와 ‘안전하고 인과적으로 입증된 사업 효과’를 분리해 검증하는 것이다.